In [1]:
import os, sys
os.environ.setdefault('CONDA_PREFIX', sys.prefix)
os.environ.get('CONDA_PREFIX')

'/opt/conda/envs/python3'

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import f1_score

In [3]:
import numpy as np
import pandas as pd

import cuml
import cupy as cp

## Preparing data for use in machine learning model

In [ ]:
data = pd.read_csv("../data/data_stemmed.csv", index_col=0)

In [ ]:
metadata = pd.read_csv("../data/995,000_rows.csv", usecols=["type", "domain"])

In [7]:
LABELS_RELIABLE = {"reliable","political","clickbait"}
LABELS_FAKE = {"fake","hate","rumor","unreliable","conspiracy","bias","junksci","satire"}

def change_label(label):
    if label in LABELS_FAKE:
        return 0
    elif label in LABELS_RELIABLE:
        return 1
    else:
        return pd.NA

In [ ]:
data["type"] = metadata["type"].apply(change_label)
data["domain"] = metadata["domain"]
data.head()

In [ ]:
data.dropna(subset=["type"], inplace=True)
data.head()

In [10]:
# Shuffles data and splits it
data = data.sample(frac=1, random_state=0)

split_index1 = int(len(data)*0.8)
split_index2 = int(len(data)*0.9)

data_training = data[:split_index1]
data_validation = data[split_index1:split_index2]
data_testing = data[split_index2:]

In [11]:
vocab_list = pd.read_csv("../data/topwords10000.csv", index_col=0)
vocab_words = vocab_list.index.to_numpy()

In [ ]:
vectorizer = CountVectorizer(vocabulary=vocab_words, dtype=np.float64)

In [13]:
X_train = vectorizer.transform(data_training["content"])
y_train = data_training["type"]

X_val = vectorizer.transform(data_validation["content"])
y_val = data_validation["type"]

X_test = vectorizer.transform(data_testing["content"])
y_test = data_testing["type"]

## Logistic regression model

In [ ]:
best_model = None
best_C = None

In [ ]:

model = cuml.LogisticRegression(max_iter=2500)

In [ ]:
model.fit(X_train, y_train)

### Model testing

In [33]:
true_y = data_testing["type"]
pred_y = model.predict(vectorizer.transform(data_testing["content"]))
pred_y = cp.asnumpy(pred_y)

In [36]:
f1_score(true_y, pred_y, pos_label=1)

0.8009733111892163